In [1]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
import os
import zipfile
from datetime import datetime

# Configurações
headers = {"User-Agent": "Mozilla/5.0"}
limite = 50
pagina = 0
todos_produtos = []
max_produtos = 10  # Limite de coleta
coletados = 0

while coletados < max_produtos:
    url = f"https://www.copafer.com.br/api/catalog_system/pub/products/search/pisos-e-revestimentos?O=OrderByReleaseDateDESC&_from={pagina * limite}&_to={(pagina + 1) * limite - 1}"
    print(f"Coletando página {pagina + 1}...")

    resposta = requests.get(url, headers=headers)
    if resposta.status_code not in [200, 206]:
        print(f"Erro ao acessar a API: {resposta.status_code}")
        break

    produtos = resposta.json()
    if not produtos:
        print("Nenhum produto encontrado. Fim da coleta.")
        break

    for produto in produtos:
        if coletados >= max_produtos:
            break
        try:
            nome = produto.get("productName", "")
            raw_link = produto.get("link", "")
            link = raw_link if raw_link.startswith("http") else "https://www.copafer.com.br" + raw_link
            codigo = produto.get("productReference", "")
            preco_promocional = produto["items"][0]["sellers"][0]["commertialOffer"]["Price"]
            preco_original = produto["items"][0]["sellers"][0]["commertialOffer"]["ListPrice"]
            preco_m2 = round(preco_promocional / produto["items"][0].get("unitMultiplier", 1), 2)

            descricao_html = produto.get("description", "")
            descricao = BeautifulSoup(descricao_html, "html.parser").get_text(separator="\n").strip()

            marca = produto.get("brand", "")
            especificacoes = f"Marca: {marca}"  # Marca incorporada em especificações

            imagem = produto["items"][0]["images"][0]["imageUrl"]
            loja = "Copafer"

            todos_produtos.append({
                "Código": codigo,
                "Nome": nome,
                "Preço Promocional": preco_promocional,
                "Preço Original": preco_original,
                "Preço por m²": preco_m2,
                "Especificações": especificacoes,
                "Descrição": descricao,
                "Imagem": imagem,
                "URL": link,
                "Loja": loja
            })

            coletados += 1

        except Exception as e:
            print(f"Erro ao processar produto: {e}")

    pagina += 1
    time.sleep(0.5)

# Pasta base e data
data_hoje = datetime.now().strftime("%Y-%m-%d")
pasta_base = "saida_copafer"
os.makedirs(pasta_base, exist_ok=True)

# Salvar CSV e Excel
csv_path = os.path.join(pasta_base, f"produtos_pisos_revestimentos_copafer_{data_hoje}.csv")
xlsx_path = os.path.join(pasta_base, f"produtos_pisos_revestimentos_copafer_{data_hoje}.xlsx")

df = pd.DataFrame(todos_produtos)
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_excel(xlsx_path, index=False)

# Baixar imagens com retry
images_dir = os.path.join(pasta_base, f"imagens_copafer_{data_hoje}")
os.makedirs(images_dir, exist_ok=True)

for _, row in df.iterrows():
    codigo = row["Código"]
    img_url = row["Imagem"]
    if img_url != "N/A":
        tentativas = 0
        sucesso = False
        while tentativas < 3 and not sucesso:
            try:
                resp = requests.get(img_url, stream=True, timeout=10)
                if resp.status_code == 200:
                    imagem_path = os.path.join(images_dir, f"{codigo}.png")
                    with open(imagem_path, "wb") as f:
                        for chunk in resp.iter_content(1024):
                            f.write(chunk)
                    sucesso = True
                else:
                    print(f"❌ Erro ao baixar imagem para {codigo}: Status {resp.status_code}")
            except Exception as e:
                print(f"❌ Erro ao baixar imagem para {codigo}: {e}")
            finally:
                if not sucesso:
                    tentativas += 1
                    if tentativas < 3:
                        print(f"🔁 Tentando novamente {codigo} (tentativa {tentativas + 1})...")
                        time.sleep(5)

# Compactar imagens em ZIP
zip_filename = os.path.join(pasta_base, f"imagens_copafer_{data_hoje}.zip")
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(images_dir):
        zipf.write(os.path.join(images_dir, filename), filename)

# Mensagens finais
print(f"📄 CSV salvo: {csv_path}")
print(f"📊 Excel salvo: {xlsx_path}")
print(f"📦 Imagens compactadas: {zip_filename}")


Coletando página 1...
📄 CSV salvo: saida_copafer/produtos_pisos_revestimentos_copafer_2025-05-28.csv
📊 Excel salvo: saida_copafer/produtos_pisos_revestimentos_copafer_2025-05-28.xlsx
📦 Imagens compactadas: saida_copafer/imagens_copafer_2025-05-28.zip
